# Rerunning the paper's experiments

This notebook retrains and redraws the experiments from *The Effects of Domain Knowledge on
Differentiable Structure Learning in Industrial Scenarios*:

1. **Figure 1**: SHD as more forbidden edges are known (`tier_mask`), on CausalMan,
   causalAssembly (station order and TierSort) and Sachs.
2. **Figure 2**: SHD as more true edges are given (`given_edges`), on CausalMan,
   causalAssembly and Sachs.
3. **Other claims**: Code is also given to demonstrate other claims from the paper: forbidden edges mostly fix orientations, given-edge recovery vs non-given edges, the effect of injecting noise on CausalMan.

The learners, grids and thresholds are the protocol in `src/train/config.toml`, only the
seeds are chosen here. The paper averaged 10 seeds (0-9), but for the sake of demonstration fewer are used here.

`JOBS` runs train at a time (several fit on one GPU). Every run is saved as it finishes,
so it is interruptible. Re-running a cell trains only what is missing. All runs go to `data/results/`, same as with `python -m train.run`, so other trained runs (e.g. from the command line on a cluster, with `--seeds` and `--shard K/N`) are reused here.

Before running, run `uv sync --extra causalassembly` and download the CausalMan data and
build the datasets with `python -m benchmarks` (see the README).

In [ ]:
from analysis import heatmap, lines, style
from analysis.diagnostics import given_edge_recovery
import benchmarks
from train import run

SEEDS = [0, 1]  # the paper used 10 seeds, list(range(10))
JOBS = 2  # runs trained at a time, each in its own process
BENCHMARKS = ["causalman", "causalassembly", "sachs"]  # remove one to skip it

RESULTS = run.RESULTS_DIR  # data/results

# Per benchmark, as in config.toml: the noise level trained on (only CausalMan gets
# injected noise) and the tier source of its main forbidden-edge panel.
NOISE = {"causalman": 0.1, "causalassembly": None, "sachs": None}
TIER = {"causalman": "reference", "causalassembly": "station", "sachs": "dag"}
TITLE = {"causalman": "CausalMan", "causalassembly": "causalAssembly", "sachs": "Sachs"}
N_EDGES = {b: benchmarks.load(b).directed_subgraph().number_of_edges() for b in BENCHMARKS}


def pick(b):
    # The runs every plot uses: z-scored data, the linear learners, noise, seeds
    return {"normalization": "zscore", "model": "linear", "noise": NOISE[b], "seed": SEEDS}


# Given-edge runs: given edges encouraged by the hinge, their reverses forbidden. Per
# point, the threshold and hinge setting (margin, weight) with the best mean SHD are used,
# and seeds whose estimate is cyclic there are scored at their smallest acyclic threshold.
HINGE = {"given_mode": "hinge", "mask_reverse": True}
HINGE_GRID = ["w_threshold", "given_hinge_margin", "given_hinge_weight"]


# Points with no knowledge (x=0) are the same for given-edge runs as masked runs (giving
# nothing = masking nothing), so we can save time by just reusing those results.
def no_knowledge(b, y):
    # The x = 0 points of the given-edge plots: tier_mask with nothing masked.
    return lines.table(
        RESULTS / b / "tier_mask",
        x="mask_fraction",
        y=y,
        where={**pick(b), "tier_source": TIER[b], "mask_fraction": 0.0},
        cyclic="fill",
    )

## Training (Figures 1 and 2)

Each grid crosses the six linear learners with the experiment's axes and the seeds:
`tier_mask` has 9 mask fractions (on causalAssembly, for both tier sources), and
`given_edges` 4 fractions given x a 3x3 hinge grid.

In [ ]:
# Train the forbidden-edge runs
for b in BENCHMARKS:
    print(f"== tier_mask on {b}")
    run.train(run.build_grid("tier_mask", b, configs=["linear"], seeds=SEEDS), jobs=JOBS)

In [ ]:
# Train the given-edge runs
for b in BENCHMARKS:
    print(f"== given_edges on {b}")
    run.train(run.build_grid("given_edges", b, configs=["linear"], seeds=SEEDS), jobs=JOBS)

## Figure 1: forbidden edges

SHD against the fraction of the forbidden edges that are masked. Per point, the threshold
with the best mean SHD is used, and a seed whose estimate is cyclic there is scored at its
own smallest acyclic threshold (`cyclic="fill"`), as in the paper. Dashed: the SHD of the
empty graph (the number of true edges). Optionally, you can enable bands which span the 
min/max of the seeds (`band=True`).

In [ ]:
FIG1 = [
    ("CausalMan", "causalman", "reference"),
    ("causalAssembly", "causalassembly", "station"),
    ("causalAssembly (TierSort)", "causalassembly", "dag"),
    ("Sachs", "sachs", "dag"),
]

for title, b, tiers in FIG1:
    if b in BENCHMARKS:
        tm = lines.table(
            RESULTS / b / "tier_mask",
            x="mask_fraction",
            y="shd",
            where={**pick(b), "tier_source": tiers},
            cyclic="fill",
        )
        lines.draw(tm, title=title, band=False, reference=N_EDGES[b])

## Figure 2: given edges

This section shows how to plot Figure 2, which is SHD over the whole graph, given edges included (`full_shd`), against the fraction of true edges given. Per point, the threshold and hinge setting with the best mean SHD are used, and a seed whose estimate is cyclic there is scored at its own smallest acyclic threshold (`cyclic="fill"`, as in Figure 1). The no-knowledge point comes from `tier_mask` with nothing masked.

In [ ]:
for b in BENCHMARKS:
    ge = lines.table(
        RESULTS / b / "given_edges",
        x="frac_given",
        y="full_shd",
        where={**pick(b), **HINGE},
        best_over=HINGE_GRID,
        cyclic="fill",
        baseline=no_knowledge(b, "full_shd"),
    )
    lines.draw(ge, title=TITLE[b], band=False, reference=N_EDGES[b])

## Other claims in the paper

Note that the first two cells here reuse the runs above, the last one requires more training.

### Forbidden edges mostly fix orientations

This plot shows the skeleton SHD (edge direction ignored) of Figure 1's runs, at the same thresholds. Where it
stays flat until nearly every forbidden edge is known, Figure 1's gains come from
correcting reversed edges rather than finding new ones.

In [ ]:
for title, b, tiers in FIG1:
    if b in BENCHMARKS:
        tm = lines.table(
            RESULTS / b / "tier_mask",
            x="mask_fraction",
            y="skel_shd",
            where={**pick(b), "tier_source": tiers},
            cyclic="fill",
        )
        lines.draw(tm, title=title, band=False, reference=N_EDGES[b])

### Given edges recovery vs non-given edges

This cell is to show that given edges are recovered more frequently than non-given edges, meaning the hinge is not completely ineffective. However, not all given edges are recovered.

This plot takes Figure 2's points and plots the TPR over the true edges that were *not* given (`tpr`), and how
many of the given edges are recovered (dashed: the number given).

In [ ]:
for b in BENCHMARKS:
    exp_dir = RESULTS / b / "given_edges"
    tpr = lines.table(
        exp_dir,
        x="frac_given",
        y="tpr",
        where={**pick(b), **HINGE},
        best_over=HINGE_GRID,
        cyclic="fill",
        baseline=no_knowledge(b, "tpr"),
    )
    lines.draw(tpr, title=f"{TITLE[b]}: edges not given", band=False)
    rec = given_edge_recovery.table(exp_dir, where={**pick(b), **HINGE}, best_over=HINGE_GRID)
    given_edge_recovery.draw(rec, title=f"{TITLE[b]}: given edges")

### CausalMan needs injected noise

This cell trains Figure 1's CausalMan grid without injected noise (`noise=[None]`), where
every method fails to recover meaningful structure, and with 5% noise (`0.05`), and uses 
the already-trained 10% noise runs (what Figure 1 uses). The noise-free runs take around four times as long as the
noisy ones, so to save GPU time only one learner (DAGMA) is trained and shown here as an
example. To include the others, train with `configs=["linear"]` and drop the
`base`/`loss_fn` filter.

The plot is one heatmap per learner: SHD by noise level (rows) and mask fraction
(columns). Each learner's threshold is chosen separately for each noise level.

In [ ]:
grid = run.build_grid("tier_mask", "causalman", configs=["dagma_adam"], noise=[None, 0.05], seeds=SEEDS)
run.train(grid, jobs=JOBS)
tm = lines.table(
    RESULTS / "causalman" / "tier_mask",
    x="mask_fraction",
    y="shd",
    where={
        **pick("causalman"),
        "base": "dagma_adam",
        "loss_fn": "l2",
        "noise": [None, 0.05, 0.1],
        "tier_source": "reference",
    },
    labels=["base", "loss_fn", "noise"],
    cyclic="fill",
)
for (base, loss_fn), line in tm.groupby(["base", "loss_fn"]):
    name = style.LINES[f"{base}/{loss_fn}"][0]
    heatmap.draw(line, rows="noise", title=f"CausalMan: {name}")

## Other settings of interest

- For nonlinear variants (MLPs), use `configs=["nonlinear"]` in the training cells and set `"model": "nonlinear"` in `pick`. Note that nonlinear causalAssembly runs take hours each.
- For plotting other metrics set `y=`. Other options and kinds of plots are described in `src/analysis/README.md`, you may be interested in consistency plots to analyze which edges are most or least consistently recovered and the degree of consistency between seeds.